In [ ]:
import os
import ast
from pathlib import Path

In [ ]:
def extract_functions_from_folder(folder_path):
    """
    Read all Python files in a folder and return top-level functions 
    (in the order they appear) for each file.
    
    Args:
        folder_path (str): Path to the folder containing Python files
        
    Returns:
        dict: Dictionary with filename as key and list of function names as value
    """
    
    folder = Path(folder_path)
    if not folder.exists():
        print(f"Folder '{folder_path}' does not exist.")
        return {}
    
    results = {}
    
    # Get all Python files in the folder
    python_files = list(folder.glob("*.py"))
    
    if not python_files:
        print(f"No Python files found in '{folder_path}'")
        return {}
    
    for py_file in python_files:
        try:
            with open(py_file, 'r', encoding='utf-8') as file:
                content = file.read()
            
            # Parse the Python file into an AST
            tree = ast.parse(content)
            
            # Extract only top-level function definitions (excluding helper functions starting with "_")
            functions = []
            for node in ast.walk(tree):
                if isinstance(node, ast.FunctionDef):
                    # Check if this is a top-level function (not nested)
                    # by checking if its parent is the module itself
                    for parent in ast.walk(tree):
                        if hasattr(parent, 'body') and node in parent.body:
                            if isinstance(parent, ast.Module):
                                # Exclude functions starting with "_" (helper/private functions)
                                if not node.name.startswith('_'):
                                    functions.append(node.name)
                            break
            
            # Remove duplicates while preserving order
            unique_functions = []
            seen = set()
            for func in functions:
                if func not in seen:
                    unique_functions.append(func)
                    seen.add(func)
            
            if unique_functions:
                results[py_file.name] = unique_functions
                
        except SyntaxError as e:
            print(f"Syntax error in {py_file.name}: {e}")
        except Exception as e:
            print(f"Error processing {py_file.name}: {e}")
    
    return results

def print_results(results):
    """
    Print the results in the specified format
    """
    if not results:
        print("No functions found or no valid Python files.")
        return
    
    for filename, functions in results.items():
        print(f"{filename}:")
        for func in functions:
            print(f"*   {func}")
        print()  # Empty line between files

In [ ]:
# Change this to your target folder path
folder_path = "../utils"

print(f"\nScanning folder: {os.path.abspath(folder_path)}")
print("-" * 50)

results = extract_functions_from_folder(folder_path)
print_results(results)

# Optionally, you can also return the raw results dictionary
print("Raw results dictionary:")
print(results)

In [ ]:
import os
import ast

In [ ]:
def extract_functions_from_folder(folder_path):
    """
    Reads all Python files in a given folder and returns a list of functions
    (in the order they appear) for each of them, excluding help, nested functions, and classes.

    Args:
        folder_path (str): The path to the folder containing the Python files.

    Returns:
        list: A list of function names extracted from the files, maintaining order.  
              Returns an empty list if the folder is invalid or no Python files are found.
    """

    all_functions = []
    if not os.path.isdir(folder_path):
        print(f"Error: Invalid folder path: {folder_path}")
        return []

    python_files = [f for f in os.listdir(folder_path) if f.endswith(".py")]

    if not python_files:
        print(f"No Python files found in: {folder_path}")
        return []
    
    for filename in python_files:
        filepath = os.path.join(folder_path, filename)
        
        try:
            with open(filepath, "r") as f:
                tree = ast.parse(f.read())
        except Exception as e:
            print(f"Error parsing file {filename}: {e}")
            continue  # Skip to the next file if parsing fails

        # Add parent attribute to nodes
        for node in ast.walk(tree):
            for child in ast.iter_child_nodes(node):
                child.parent = node

        for node in ast.walk(tree):
            if isinstance(node, ast.FunctionDef):
                # Exclude nested functions (functions defined inside other functions)
                if node.parent is not None and isinstance(node.parent, ast.FunctionDef):
                    continue
                #Exlcude classes
                if node.parent is not None and isinstance(node.parent, ast.ClassDef):
                    continue
                # Exclude helper functions (starting with '_')
                if node.name.startswith("_"):
                    continue

                all_functions.append(node.name)

    return all_functions

In [ ]:
# Example usage (in Jupyter Notebook):
folder_path = "../utils"  # Replace with your folder path

functions = extract_functions_from_folder(folder_path)

# Format the output as requested
function_dict = {}
for filename in [f for f in os.listdir(folder_path) if f.endswith(".py")]:
    function_dict[filename] = []

for func in functions:
    for filename in [f for f in os.listdir(folder_path) if f.endswith(".py")]:
        filepath = os.path.join(folder_path, filename)
        try:
            with open(filepath, "r") as f:
                if func in f.read():
                    function_dict[filename].append(func)
                    break # Ensure the function is only associated with the first file it's found in
        except Exception as e:
            print(f"Error reading file {filename}: {e}")

# Print the formatted output
for filename, funcs in function_dict.items():
    print(f"{filename}:")
    for func in funcs:
        print(f"*   {func}")